# Trabajo de Laboratorio N° 2: Sistemas Inteligentes
## NLP – Representaciones Vectoriales – RAG – Agentes Basados en LLMs
* **Dominio Técnico:** GitHub REST API (Especificación v3 / Versión 2022-11-28)
* **Cátedra:** Inteligencia Artificial – FCEFyN
* **Alcance de este Notebook:** 
  * **Fase 2:** Construcción del Corpus, Estrategia de Chunking, Embeddings e Indexación Vectorial (FAISS) y Validación de Búsqueda Semántica (*Retrieval*).
  * **Fase 3 (Parte 1 - Conexión e Integración):** Vinculación del motor de recuperación vectorial con el LLM mediante LangChain Expression Language (LCEL) para generación de respuestas fundamentadas.

## 1. Instalación e Importación de Dependencias
Instalamos las librerías necesarias para el procesamiento de texto, base vectorial FAISS y clientes de modelos de lenguaje.

In [ ]:
# Instalación de librerías requeridas (ejecutar en Google Colab o entorno nuevo)
!pip install -q langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu sentence-transformers requests

In [ ]:
import os
import glob
from getpass import getpass
from typing import List, Dict, Any

# LangChain y componentes de RAG
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

print("✅ Librerías importadas correctamente.")

## 2. Fase 2: Carga del Corpus Técnico y Segmentación (Chunking)
Cargamos los documentos técnicos formalizados en `./corpus`:
1. `01_autenticacion_y_rate_limits.md`
2. `02_repositorios_y_ramas.md`
3. `03_issues_y_etiquetas.md`
4. `04_pull_requests_y_revisiones.md`

Aplicamos `RecursiveCharacterTextSplitter` para segmentar los documentos preservando la coherencia semántica de encabezados, tablas y bloques de código de endpoints.

In [ ]:
# 1. Localización de los archivos del corpus
CORPUS_DIR = "./corpus"
md_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.md")))
tech_files = [f for f in md_files if not os.path.basename(f).lower().startswith("readme")]

print(f"📄 Archivos técnicos encontrados ({len(tech_files)}):")
for f in tech_files:
    print(f"  - {f}")

# 2. Carga con TextLoader
raw_documents = []
for filepath in tech_files:
    loader = TextLoader(filepath, encoding="utf-8")
    raw_documents.extend(loader.load())

print(f"\n✅ Total de documentos originales cargados: {len(raw_documents)}")

# 3. Segmentación (Chunking) semántica
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(raw_documents)
print(f"✅ Documentos divididos en {len(chunks)} chunks.")

# 4. Inspección de un chunk representativo
print("\n🔍 Ejemplo de chunk generado:")
print(f"Fuente: {chunks[0].metadata['source']}")
print("-" * 50)
print(chunks[0].page_content[:350] + "...")

## 3. Fase 2: Generación de Embeddings e Indexación Vectorial (FAISS)
Convertimos cada chunk en una representación vectorial utilizando el modelo de Hugging Face (`sentence-transformers/all-MiniLM-L6-v2`) y los almacenamos en un índice vectorial **FAISS** optimizado para búsquedas por similitud de cosenos.

In [ ]:
# Selección del modelo de embeddings
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"⏳ Inicializando modelo de embeddings: {EMBEDDING_MODEL_NAME}...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Construcción de la base vectorial FAISS
print("⏳ Construyendo índice vectorial FAISS a partir de los chunks...")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding_model)

# Persistencia local opcional
vectorstore.save_local("faiss_github_api_index")
print("✅ Índice vectorial FAISS creado y guardado en disco exitosamente.")

## 4. Fase 2: Motor de Recuperación (Retrieval) y Validación Semántica
Configuramos el `retriever` para recuperar los 3 fragmentos más cercanos a la consulta del usuario y testeamos la recuperación de información técnica pura antes de pasársela al LLM.

In [ ]:
# Configuración del Retriever (Top-3 por similitud de cosenos)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

def probar_retrieval(consulta: str):
    print(f"\n❓ Consulta de prueba: '{consulta}'")
    docs = retriever.invoke(consulta)
    print(f"📄 Fragmentos recuperados: {len(docs)}")
    for i, doc in enumerate(docs, 1):
        origen = os.path.basename(doc.metadata.get('source', 'desconocido'))
        print(f"\n  [Chunk {i} | Fuente: {origen}]")
        print("  " + doc.page_content.strip()[:250].replace('\n', '\n  ') + "...")
    print("=" * 60)

# Validaciones semánticas del motor de recuperación
probar_retrieval("¿Cuáles son los encabezados obligatorios y qué pasa si no se envía User-Agent?")
probar_retrieval("¿Cómo crear un issue y qué parámetros son requeridos?")
probar_retrieval("¿Cuáles son los métodos disponibles para hacer merge de un Pull Request?")

## 5. Fase 3: Conexión e Integración (Vincular Retriever con LLM)
En este bloque completamos la etapa de **Conexión e Integración** requerida por la Fase 3 de la consigna:
> *"Vincular el motor de recuperación vectorial con un modelo generativo de lenguaje (LLM de acceso abierto local o a través de una API)."*

### Arquitectura de Integración (RAG Pipeline):
```text
           [ Consulta del Usuario ]
                      │
        ┌─────────────┴─────────────┐
        ▼                           ▼
[ Retriever (FAISS) ]     [ Prompt Template ]
        │ (Top-3 Chunks)            │
        └─────────────►┬◄───────────┘
                       │ Contexto + Pregunta
                       ▼
            [ LLM (Qwen / Llama) ]
                       │
                       ▼
         [ Respuesta Fundamentada ]
```

In [ ]:
# Conexión con el LLM (Hugging Face Router / OpenAI / Groq)
# Se utiliza el endpoint compatible de OpenAI con HF Router (usado por la cátedra)
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Ingresá tu HF_TOKEN (Hugging Face): ")

llm = ChatOpenAI(
    api_key=os.environ["HF_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    model="Qwen/Qwen2.5-7B-Instruct",
    temperature=0.1
)

print(f"✅ Conexión con el LLM establecida correctamente ({llm.model_name}).")

In [ ]:
# Formateador de contexto con trazabilidad de fuentes
def formatear_contexto(docs: List[Any]) -> str:
    bloques = []
    for i, doc in enumerate(docs, 1):
        fuente = os.path.basename(doc.metadata.get("source", "desconocido"))
        bloques.append(f"--- [FRAGMENTO {i} | FUENTE: {fuente}] ---\n{doc.page_content.strip()}")
    return "\n\n".join(bloques)

# Prompt del Sistema y Plantilla para RAG
template_rag = """Eres un Asistente Técnico Senior especializado en la API REST de GitHub (v3).
Tu tarea es responder preguntas técnicas con máxima precisión basándote ÚNICAMENTE en la documentación oficial provista en el CONTEXTO.

Instrucciones obligatorias:
1. Detalla métodos HTTP, rutas completas, códigos de estado, parámetros y cabeceras según corresponda.
2. Incluye siempre la cita de la fuente utilizada (por ejemplo: [Fuente: 01_autenticacion_y_rate_limits.md]).
3. Si la información no se encuentra en el contexto provisto, responde exactamente: "La documentación provista en el corpus no contiene información suficiente para responder a esta consulta." No agregues suposiciones.

CONTEXTO TÉCNICO OFICIAL:
{contexto}

PREGUNTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA FUNDAMENTADA:"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

# Pipeline LCEL (LangChain Expression Language) para Conexión e Integración
cadena_rag = (
    {
        "contexto": retriever | formatear_contexto,
        "pregunta": RunnablePassthrough()
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG (Retriever + Prompt + LLM) conectado e integrado.")

## 6. Pruebas de Interacción RAG Fundamentada
Ejecutamos consultas técnicas para comprobar la integración de la cadena completa (recuperación + síntesis sin alucinaciones).

In [ ]:
def consultar_asistente_rag(pregunta: str) -> str:
    print("=" * 70)
    print(f"🧑‍💻 CONSULTA: {pregunta}")
    print("=" * 70)
    respuesta = cadena_rag.invoke(pregunta)
    print(f"🤖 RESPUESTA:\n{respuesta}\n")
    return respuesta

# 1. Consulta sobre Autenticación y Cabeceras
r1 = consultar_asistente_rag(
    "¿Cuáles son los encabezados HTTP obligatorios para hacer peticiones a la API de GitHub y qué ocurre si no envío el User-Agent?"
)

# 2. Consulta sobre Pull Requests y Parámetros
r2 = consultar_asistente_rag(
    "¿Qué método y endpoint debo usar para crear un Pull Request y cuáles son los campos obligatorios del cuerpo JSON?"
)

# 3. Consulta sobre Rate Limiting y Cuotas
r3 = consultar_asistente_rag(
    "¿Cuál es el límite de solicitudes por hora para usuarios no autenticados frente a usuarios autenticados, y qué código HTTP devuelve GitHub al excederlo?"
)